<a href="https://colab.research.google.com/github/ChiriKamau/limaAI/blob/main/notebooks/Quantize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Start

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# Install the YOLOv8 library
!pip install ultralytics

ValueError: Mountpoint must not already contain files

In [4]:
import shutil
import os
from ultralytics import YOLO

# --- CONFIGURATION ---
# The folder where your .pt model lives currently
model_parent_dir = '/content/drive/MyDrive/Tomato_dataset/models/fruit_model_pt/' # Corrected to be the directory
model_filename = 'best_large_model.pt' # The actual .pt model file name

# The NEW folder where we want to save the result
drive_quant_dir = os.path.join(model_parent_dir, 'Kaggle_quant') # Now Kaggle_quant will be created inside model_parent_dir

# --- EXECUTION ---

# 1. Create the new "quantized" folder in Drive if it doesn't exist
os.makedirs(drive_quant_dir, exist_ok=True)
print(f"📂 Target folder confirmed: {drive_quant_dir}")

# 2. Copy the model FROM Drive TO Colab local storage (Safe zone)
drive_model_path = os.path.join(model_parent_dir, model_filename) # Corrected to use model_parent_dir and model_filename
local_model_name = model_filename

print(f"🔄 Copying model to local workspace...")
shutil.copy(drive_model_path, local_model_name)

# 3. Load and Export locally
print("⏳ Starting export (this happens locally)...")
model = YOLO(local_model_name)
model.export(format='tflite', int8=True)

# 4. Move the result BACK to the new "quantized" folder in Drive
# Ultralytics creates a folder named like 'yolov8m_fruits_saved_model' or 'best_large_model_saved_model'
# We need to ensure local_result_folder matches what YOLO creates, which is usually based on the model name.
local_result_folder = model_filename.replace('.pt', '_saved_model') # This should correctly get the folder name
final_destination = os.path.join(drive_quant_dir, local_result_folder)

print(f"💾 Moving results to: {final_destination}")

# Clean up old version if it exists in the destination to avoid errors
if os.path.exists(final_destination):
    shutil.rmtree(final_destination)

# Move the folder
if os.path.exists(local_result_folder):
    shutil.copytree(local_result_folder, final_destination)
    print("✅ Success! Check your 'quantized' folder in Google Drive.")
else:
    print("❌ Error: The local export folder was not found.")

📂 Target folder confirmed: /content/drive/MyDrive/Tomato_dataset/models/fruit_model_pt/Kaggle_quant
🔄 Copying model to local workspace...
⏳ Starting export (this happens locally)...
Ultralytics 8.3.248 🚀 Python-3.12.12 torch-2.9.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
WARNING ⚠️ INT8 export requires a missing 'data' arg for calibration. Using default 'data=coco8.yaml'.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 112 layers, 43,608,150 parameters, 0 gradients, 164.8 GFLOPs

PyTorch: starting from 'best_large_model.pt' with input shape (1, 3, 416, 416) BCHW and output shape(s) (1, 6, 3549) (83.5 MB)
requirements: Ultralytics requirements ['sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0', 'onnx>=1.12.0,<2.0.0', 'onnx2tf>=1.26.3', 'onnxslim>=0.1.71', 'onnxruntime'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 

/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:1447: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  warnings.warn(


ONNX: slimming with onnxslim 0.1.82...
ONNX: export success ✅ 6.7s, saved as 'best_large_model.onnx' (166.6 MB)
Unzipping calibration_image_sample_data_20x128x128x3_float32.npy.zip to /content/calibration_image_sample_data_20x128x128x3_float32.npy...: 100% ━━━━━━━━━━━━ 1/1 25.7files/s 0.0s
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at 'best_large_model_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 416, 416, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 6, 3549), dtype=tf.float32, name=None)
Captures:
  133709462165776: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  133709462164240: TensorSpec(shape=(3, 3, 3, 64), dtype=tf.float32, name=None)
  133709462165008: TensorSpec(shape=(64,), dtype=tf.float32, name=None)
  133709462169232: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  133709462170384: TensorSpec(shape=(3, 3, 6

In [6]:
import shutil
import os
import tensorflow as tf

# --- CONFIGURATION ---
# 1. The exact path you provided
source_model_path = '/content/drive/MyDrive/Tomato_dataset/models/green_mobilenet(v3)/green_mobilenet(v3.5).keras'

# 2. Output directory in Drive
drive_quant_dir = os.path.dirname(source_model_path) # We will save it in the same folder as the original
output_filename = 'ripe_mobilenet_v3.3_quantized.tflite'

# --- EXECUTION ---

print(f"📂 Source Model: {source_model_path}")

# 1. Copy model to local Colab storage (faster processing & safer)
local_model_name = 'temp_model.keras'
print("🔄 Copying model to local workspace...")
shutil.copy(source_model_path, local_model_name)

# 2. Load and Quantize
print("⏳ Starting conversion (Enabling Flex Ops)...")

try:
    # Load the Keras model
    model = tf.keras.models.load_model(local_model_name)

    # Initialize Converter
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # --- 🛠️ CRITICAL FIX ---
    # This enables the "Select TF Ops" (Flex) delegate.
    # It allows TFLite to use the original TensorFlow implementation for layers
    # that it doesn't have a built-in optimized version for (like your specific Conv2D/Relu6 blocks).
    converter.target_spec.supported_ops = [
      tf.lite.OpsSet.TFLITE_BUILTINS, # Try standard TFLite first
      tf.lite.OpsSet.SELECT_TF_OPS    # Fallback to TensorFlow Ops if needed
    ]

    # Optimization (makes it smaller)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    # Convert
    tflite_model = converter.convert()

    # Save locally
    with open(output_filename, 'wb') as f:
        f.write(tflite_model)

    print(f"✅ Conversion successful: {output_filename}")

    # 3. Move result back to Drive
    final_path = os.path.join(drive_quant_dir, output_filename)
    shutil.copy(output_filename, final_path)

    print(f"💾 Saved to Google Drive: {final_path}")

except Exception as e:
    print(f"❌ Error during conversion: {e}")

📂 Source Model: /content/drive/MyDrive/Tomato_dataset/models/green_mobilenet(v3)/green_mobilenet(v3.5).keras
🔄 Copying model to local workspace...
⏳ Starting conversion (Enabling Flex Ops)...


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 10 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Saved artifact at '/tmp/tmpq1t61uqh'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name='input_layer_9')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float16, name=None)
Captures:
  139507045205392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139507037387280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139507037387472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139507037387856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139507037388432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139507037386896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139507037386512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139507037386320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139507037384784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139507037385744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1395070373876

In [1]:
import tensorflow as tf
import os

# --- PATH CONFIGURATION ---
# UPDATE THESE PATHS to match where your .keras models are in Google Drive
ripe_source_path  = '/content/drive/MyDrive/Tomato_dataset/models/ripe_mobilenet(v3)/ripe_mobilenet(v3.3).keras'
green_source_path = '/content/drive/MyDrive/Tomato_dataset/models/green_mobilenet(v3)/green_mobilenet(v3.5).keras' # Update this path!

# Output folder
output_dir = '/content/drive/MyDrive/Tomato_dataset/models/Standard_Quant/'
os.makedirs(output_dir, exist_ok=True)

def convert_to_standard_tflite(name, keras_path):
    print(f"\n🔄 Processing: {name}...")

    if not os.path.exists(keras_path):
        print(f"❌ File not found: {keras_path}")
        return

    try:
        # 1. Load Keras Model
        model = tf.keras.models.load_model(keras_path)

        # 2. FORCE CONCRETE FUNCTION (The Fix for Flex Error)
        # This tells TFLite: "Input will ALWAYS be 1 image of 256x256"
        # This removes dynamic ops that cause the crash.
        run_model = tf.function(lambda x: model(x))
        concrete_func = run_model.get_concrete_function(
            tf.TensorSpec(shape=[1, 256, 256, 3], dtype=tf.float32)
        )

        # 3. Convert
        converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

        # IMPORTANT: We do NOT use SELECT_TF_OPS here.
        # We want pure, standard TFLite.

        tflite_model = converter.convert()

        # 4. Save
        save_path = os.path.join(output_dir, f"{name}_standard.tflite")
        with open(save_path, 'wb') as f:
            f.write(tflite_model)

        print(f"✅ Success! CLEAN model saved: {save_path}")

    except Exception as e:
        print(f"❌ Error converting {name}: {e}")

# Run the conversion
convert_to_standard_tflite("ripe_mobilenet_v3.3", ripe_source_path)
convert_to_standard_tflite("green_mobilenet", green_source_path)


🔄 Processing: ripe_mobilenet_v3.3...
❌ File not found: /content/drive/MyDrive/Tomato_dataset/models/ripe_mobilenet(v3)/ripe_mobilenet(v3.3).keras

🔄 Processing: green_mobilenet...
❌ File not found: /content/drive/MyDrive/Tomato_dataset/models/green_mobilenet(v3)/green_mobilenet(v3.5).keras
